In [1]:
import sys

import pandas as pd

import torch
from torch.utils.data import DataLoader

from sklearn.model_selection import train_test_split

from utils.general_utils import set_stdout_to_file, set_seed

from config.feature_config import FeatureConfig

from model.preprocessor import PreprocessorArtifacts

from dice4el.scenario.scenario_handler import ScenarioHandler
from dice4el.scenario.scenario_model import ScenarioLSTM, train_ScenarioLSTM, validate_ScenarioLSTM

### --- Load Dataset ---

In [2]:
set_seed(seed=42)

In [3]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
device

device(type='cuda')

In [4]:
df = pd.read_excel(
    "../../../data/bpic12.xlsx",
    engine="openpyxl",
    keep_default_na=False,
    dtype={
        "case:concept:name": "string",
        "concept:name": "string",
        "lifecycle:transition": "string",
        "org:resource": "string",
        "case:REG_DATE_HR": "string",
        "case:REG_DATE_DAY": "string",
        "case:REG_DATE_MON": "string",
        "case:AMOUNT_REQ": "float32",
        "time_delta": "float32",
    }
)

df["time:timestamp"] = pd.to_datetime(df["time:timestamp"])

In [5]:
df.head(20)

,case:concept:name,time:timestamp,case:AMOUNT_REQ,case:REG_DATE_DAY,case:REG_DATE_HR,case:REG_DATE_MON,concept:name,lifecycle:transition,org:resource,time_delta
0,173688,2011-10-01 00:38:44.546,20000.0,Saturday,12 AM,October,A_SUBMITTED,COMPLETE,112,0.000000
1,173688,2011-10-01 00:38:44.880,20000.0,Saturday,12 AM,October,A_PARTLYSUBMITTED,COMPLETE,112,0.334000
2,173688,2011-10-01 00:39:37.906,20000.0,Saturday,12 AM,October,A_PREACCEPTED,COMPLETE,112,53.026001
3,173688,2011-10-01 11:42:43.308,20000.0,Saturday,12 AM,October,A_ACCEPTED,COMPLETE,10862,39785.402344
4,173688,2011-10-01 11:45:09.243,20000.0,Saturday,12 AM,October,O_SELECTED,COMPLETE,10862,145.934998
5,173688,2011-10-01 11:45:09.243,20000.0,Saturday,12 AM,October,A_FINALIZED,COMPLETE,10862,0.000000
6,173688,2011-10-01 11:45:11.197,20000.0,Saturday,12 AM,October,O_CREATED,COMPLETE,10862,1.954000
7,173688,2011-10-01 11:45:11.380,20000.0,Saturday,12 AM,October,O_SENT,COMPLETE,10862,0.183000
8,173688,2011-10-10 11:33:03.668,20000.0,Saturday,12 AM,October,O_SENT_BACK,COMPLETE,11049,776872.312500
9,173688,2011-10-13 10:37:29.226,20000.0,Saturday,12 AM,October,A_REGISTERED,COMPLETE,10629,255865.562500


In [7]:
feature_config = FeatureConfig.load(
    path = "../pretrained_models/"
)
feature_config.summary()

  activity_feature:    concept:name
  feature_order:       ['case:AMOUNT_REQ', 'case:REG_DATE_DAY', 'case:REG_DATE_HR', 'case:REG_DATE_MON', 'concept:name', 'lifecycle:transition', 'org:resource', 'time_delta']
--------------------------------------------------------------------------------------------------------------------------------------
Feature                        Type           Level    Vary   Range/Categories                         MAD        Source              
--------------------------------------------------------------------------------------------------------------------------------------
time_delta                     continuous     event    yes    [0.00, 1315.95]                          0.4980     quantile_derived    
case:AMOUNT_REQ                continuous     case     yes    [3000.00, 40000.00]                      5000.0000  quantile_derived    
case:REG_DATE_DAY              categorical    case     yes    ['Friday', 'Monday', 'Saturday', ...]    N/A        

In [8]:
preprocessor_artifacts = PreprocessorArtifacts.load(
    path = "../pretrained_models/"
)

### --- Scenario Model ---

In [9]:
scenario_handler =  ScenarioHandler(
    feature_config=feature_config,
    preprocessor_artifacts=preprocessor_artifacts
)

In [10]:
scenario_df = scenario_handler.generate_scenario_df(
    df=df,
    case_id_field="case:concept:name",
    sort_field="time:timestamp",
    n_scenarios_per_length=3
)

In [11]:
scenario_df.head()

,case:concept:name,time_index,fake,case:AMOUNT_REQ,case:REG_DATE_DAY,case:REG_DATE_HR,case:REG_DATE_MON,concept:name,lifecycle:transition,org:resource,time_delta
0,205319,0,False,5000.0,Tuesday,03 PM,January,A_SUBMITTED,COMPLETE,112,0.000000
1,205319,1,False,5000.0,Tuesday,03 PM,January,A_PARTLYSUBMITTED,COMPLETE,112,1.441000
2,205319,2,False,5000.0,Tuesday,03 PM,January,A_PREACCEPTED,COMPLETE,112,43.291000
3,205319,3,False,5000.0,Tuesday,03 PM,January,A_ACCEPTED,COMPLETE,11003,8686.115234
4,205319,4,False,5000.0,Tuesday,03 PM,January,A_FINALIZED,COMPLETE,11003,183.248001


In [12]:
case_ids = scenario_df["case:concept:name"].unique()

train_cases, val_cases = train_test_split(case_ids, test_size=0.2, random_state=42)

train_df = scenario_df[scenario_df["case:concept:name"].isin(train_cases)].copy()
val_df   = scenario_df[scenario_df["case:concept:name"].isin(val_cases)].copy()

In [13]:
train_dataset = scenario_handler.transform_dataframe_to_scenariodataset(
    scenario_df=train_df,
    case_id_field="case:concept:name",
    sort_field="time_index"
)

In [14]:
val_dataset = scenario_handler.transform_dataframe_to_scenariodataset(
    scenario_df=val_df,
    case_id_field="case:concept:name",
    sort_field="time_index"
)

In [15]:
train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=32, shuffle=False)

In [16]:
criterion = torch.nn.BCEWithLogitsLoss()

In [17]:
log_file, original_stdout = set_stdout_to_file(filepath="logs/bpic12-scenario_model_output.txt")

Epoch 020/100 | Train Loss: 0.0001 | LR: 9.05e-04
Epoch 040/100 | Train Loss: 0.0000 | LR: 6.55e-04
Epoch 060/100 | Train Loss: 0.0000 | LR: 3.46e-04
Epoch 080/100 | Train Loss: 0.0000 | LR: 9.64e-05
Epoch 100/100 | Train Loss: 0.0000 | LR: 1.00e-06
Time taken for scenario model (training): 4542.155144 seconds
Time taken for scenario model (validation): 3.655853 seconds
Val loss: {'loss': 0.00011747839183596487, 'accuracy': 0.9999864551869861, 'f1_macro': 0.9999816500642822, 'f1_weighted': 0.9999864550597991}


In [18]:
embedding_metadata = scenario_handler.get_scenario_embedding_metadata()

scenario_model = ScenarioLSTM(
    categorical_info=embedding_metadata["categorical_info"],
    n_continuous=embedding_metadata["n_continuous"],
    n_classes=embedding_metadata["n_classes"]
)

train_loss_history = train_ScenarioLSTM(
    model=scenario_model,
    train_loader=train_loader,
    learning_rate=1e-3,
    criterion=criterion,
    num_epochs=100,
    device=device
)

scenario_model.save()

In [19]:
# scenario_model = ScenarioLSTM.load()

In [20]:
val_loss = validate_ScenarioLSTM(
    model=scenario_model,
    val_loader=val_loader,
    criterion=criterion,
    device=device
)

print("Val loss:", val_loss)

In [21]:
from dice4el.scenario.scenario_model_wrapper import ScenarioModelWrapper

scenario_model_wrapper = ScenarioModelWrapper(
    scenario_model=scenario_model,
    scenario_handler=scenario_handler,
    device=device
)

In [23]:
first_case_id = scenario_df["case:concept:name"].iloc[0]
first_case = scenario_df[scenario_df["case:concept:name"] == first_case_id].sort_values("time_index")
first_case = first_case.iloc[:4]
prefix = first_case[list(feature_config.feature_order)].to_numpy()

In [24]:
prefix

array([[5000.0, 'Tuesday', '03 PM', 'January', 'A_SUBMITTED', 'COMPLETE',
        '112', 0.0],
       [5000.0, 'Tuesday', '03 PM', 'January', 'A_PARTLYSUBMITTED',
        'COMPLETE', '112', 1.440999984741211],
       [5000.0, 'Tuesday', '03 PM', 'January', 'A_PREACCEPTED',
        'COMPLETE', '112', 43.29100036621094],
       [5000.0, 'Tuesday', '03 PM', 'January', 'A_ACCEPTED', 'COMPLETE',
        '11003', 8686.115234375]], dtype=object)

In [26]:
logits = scenario_model_wrapper.simulate_forward(
    prefix=prefix,
)

In [27]:
logits

array([-22.459393], dtype=float32)

### --- Cleanup ---

In [28]:
sys.stdout = original_stdout
log_file.close()